# Proteins

This notebook embeds real proteins, fetches real protein annotations, and uses embpy plotting utilities to inspect and compare embedding spaces.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import BioEmbedder, pl, tl

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


proteins = ["TP53", "EGFR", "BRCA1", "EGF", "STAT1", "JUN", "CDK1", "IRF1"]
print(f"Protein identifiers: {proteins}")

protein_space = ad.AnnData(
    X=np.zeros((len(proteins), 1), dtype=np.float32),
    obs=pd.DataFrame(
        {"symbol": proteins},
        index=pd.Index(proteins, name="protein_symbol"),
    ),
    var=pd.DataFrame(index=["placeholder_feature"]),
)

display(protein_space)


## Embed proteins

Here proteins are rows of a small AnnData with fake `.X` and real protein symbols. We attach embeddings to `.obsm` because the rest of this tutorial analyzes proteins as observations; feature-level protein embeddings still default to `.varm`.


In [ ]:
for model_name, key in [
    ("esm2_8M", "X_esm2_8M"),
    ("prot_t5_xl_half", "X_prot_t5_xl_half"),
]:
    protein_space = embedder.embed(
        protein_space,
        entity_type="protein",
        id_type="symbol",
        obs_column="symbol",
        model=model_name,
        output="anndata",
        attach_to="obs",
        key=key,
        pooling_strategy="mean",
        show_progress=True,
    )

protein_space.obs["protein_id"] = protein_space.obs_names.astype(str)
if "gene_symbol" in protein_space.obs:
    protein_space.obs["symbol"] = protein_space.obs["gene_symbol"].fillna(protein_space.obs["symbol"])

display(protein_space)
print("obsm keys for plotting:", list(protein_space.obsm.keys()))


## Annotate proteins

`tl.annotate_proteins` queries the package protein annotation layer and writes compact summaries to `.obs`, with full records in `.uns`.


In [ ]:
protein_space = tl.annotate_proteins(
    protein_space,
    column="symbol",
    id_type="auto",
    sources=["metadata", "function", "location", "domains", "ptms", "diseases", "go", "interactions", "isoforms"],
    copy=True,
)

display(compact_obs(protein_space, ("prot_",), base=["symbol", "protein_id"]))
print("annotation stores:", [k for k in protein_space.uns if "annotation" in k])


## Plot annotated protein spaces


In [ ]:
color_key = "prot_location" if "prot_location" in protein_space.obs else "symbol"
pl.plot_embedding_space(
    protein_space,
    obsm_key="X_esm2_8M",
    method="pca",
    color=color_key,
    annotate=True,
    annotate_col="symbol",
    title="ESM-2 protein embeddings colored by UniProt location",
)

if "prot_n_domains" in protein_space.obs:
    pl.plot_embedding_space(
        protein_space,
        obsm_key="X_prot_t5_xl_half",
        method="pca",
        color="prot_n_domains",
        annotate=True,
        annotate_col="symbol",
        title="ProtT5 protein embeddings colored by domain count",
    )


## Compare protein models


In [ ]:
k = min(3, protein_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(protein_space, "X_esm2_8M", "X_prot_t5_xl_half", k=k)
print(f"Mean ESM/ProtT5 KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(protein_space, obsm_keys=["X_esm2_8M", "X_prot_t5_xl_half"], k=k)
pl.cross_embedding_correlation(protein_space, "X_esm2_8M", "X_prot_t5_xl_half")
pl.embedding_distributions(protein_space, obsm_keys=["X_esm2_8M", "X_prot_t5_xl_half"])


## Save a reusable artifact


In [ ]:
protein_space.write_h5ad(OUTPUT_DIR / "protein_embeddings.h5ad")
print(OUTPUT_DIR / "protein_embeddings.h5ad")
